# Stage 10: Pinecone Cloud Upload

In [ ]:
!pip install -q pinecone-client torch

In [ ]:
import torch
import json
import os
import time
from pinecone import Pinecone, ServerlessSpec
from kaggle_secrets import UserSecretsClient

MODELS_DIR = '/kaggle/input/datasets/avigyanray/dankgpt-models-baai'

if not os.path.exists(os.path.join(MODELS_DIR, 'embeddings_official.pt')):
    print("Searching for databases in /kaggle/input...")
    import glob
    files = glob.glob('/kaggle/input/**/embeddings_official.pt', recursive=True)
    if files:
        MODELS_DIR = os.path.dirname(files[0])

print(f"Loading databases from {MODELS_DIR}...")
db_official_emb = torch.load(os.path.join(MODELS_DIR, 'embeddings_official.pt'))
with open(os.path.join(MODELS_DIR, 'metadata_official.json'), 'r', encoding='utf-8') as f:
    db_official_meta = json.load(f)
    
db_community_emb = torch.load(os.path.join(MODELS_DIR, 'embeddings_community.pt'))
with open(os.path.join(MODELS_DIR, 'metadata_community.json'), 'r', encoding='utf-8') as f:
    db_community_meta = json.load(f)

print("Databases loaded into RAM successfully!")

In [ ]:
# 2. Connect to Pinecone
user_secrets = UserSecretsClient()
pinecone_key = user_secrets.get_secret("PINECONE_KEY")
pc = Pinecone(api_key=pinecone_key)

INDEX_NAME = "dankgpt"

# Create Index if it doesn't exist
if INDEX_NAME not in [i.name for i in pc.list_indexes()]:
    print(f"Creating index '{INDEX_NAME}'... (This takes about 30 seconds)")
    pc.create_index(
        name=INDEX_NAME,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(INDEX_NAME)
print("Connected to Pinecone Index!")

In [ ]:
# 3. Helper Function to Batch Upload
def upload_to_pinecone(embeddings_tensor, metadata_list, namespace, batch_size=200):
    print(f"\nStarting upload to namespace: '{namespace}'...")
    total_vectors = len(metadata_list)
    
    vectors_to_upsert = []
    for i in range(total_vectors):
        # Format for Pinecone: (id, vector_list, metadata_dict)
        vector_id = f"{namespace}_{i}"
        vector_values = embeddings_tensor[i].tolist()
        
        # Pinecone metadata must be key-value pairs (strings, numbers, bools, or lists of strings)
        raw_data = metadata_list[i].get('raw_data', metadata_list[i].get('knowledge', ''))
        raw_string = json.dumps(raw_data) if isinstance(raw_data, dict) else str(raw_data)
        
        meta = {
            "topic": metadata_list[i].get("topic", ""),
            "category": metadata_list[i].get("category", ""),
            "raw_data": raw_string
        }
        
        vectors_to_upsert.append((vector_id, vector_values, meta))
        
        # Upload in batches
        if len(vectors_to_upsert) >= batch_size:
            index.upsert(vectors=vectors_to_upsert, namespace=namespace)
            print(f"Uploaded {i+1}/{total_vectors} vectors to '{namespace}'...")
            vectors_to_upsert = []
            time.sleep(0.5) # Prevent rate limiting
            
    # Upload remaining
    if vectors_to_upsert:
        index.upsert(vectors=vectors_to_upsert, namespace=namespace)
        print(f"Uploaded {total_vectors}/{total_vectors} vectors to '{namespace}'...")
        
    print(f"Finished uploading {namespace}!")

# 4. Upload Official Brain
upload_to_pinecone(db_official_emb, db_official_meta, namespace="official")

# 5. Upload Community Brain
upload_to_pinecone(db_community_emb, db_community_meta, namespace="community")

print("\n\nALL DATABASES SUCCESSFULLY MIGRATED TO PINECONE CLOUD!")